# Word Embeddings

Welcome to this complete, self-contained solution notebook on **text preprocessing** and **word embeddings**.

In this assignment, we will work with book metadata such as titles, authors, descriptions, genres, and places. Our goal is to transform this raw text into meaningful numerical representations that can capture relationships between words.

---

## 🎯 Assignment Goal

The main objective is to train, test, visualize, and compare different word embedding models.

We will implement and explore three approaches:

### 1. Word2Vec from Scratch
We will build a simple **Skip-Gram Word2Vec model** using negative sampling.  
This helps us understand how word embeddings are learned internally.

### 2. FastText with Gensim
We will train a FastText model using the `gensim` library.  
FastText improves on Word2Vec by learning from subword information, which helps with rare or unseen words.

### 3. FastText with the Native `fasttext` Package
We will also use the official `fasttext` package to train embeddings and compare its behavior with the Gensim implementation.

---

## 🧠 What You Will Learn

By the end of this notebook, you will be able to:

- clean and normalize raw book metadata,
- convert text into a training corpus,
- build a vocabulary from tokenized text,
- generate Skip-Gram training pairs,
- train word embeddings from scratch,
- train FastText embeddings using two different libraries,
- test semantic relationships using simple analogy examples,
- visualize learned word vectors in two dimensions,
- compare the strengths and limitations of Word2Vec and FastText.

---

## 🧩 Final Outcome

At the end of the notebook, we will have multiple trained embedding models and a clearer understanding of how different embedding techniques represent words, authors, genres, and places in a shared vector space.

## 1. Environment Setup

The notebook contains all assignment code directly in notebook cells.

Some libraries may already be installed in your environment. If a package is missing, run the optional installation cell below.

In [ ]:
RUN_INSTALLS = True

if RUN_INSTALLS:
    import importlib.util
    import subprocess
    import sys

    package_map = {
        "pandas": "pandas",
        "numpy": "numpy",
        "matplotlib": "matplotlib",
        "nltk": "nltk",
        "gensim": "gensim",
        "tqdm": "tqdm",
        "fasttext": "fasttext",
    }

    for module_name, package_name in package_map.items():
        if importlib.util.find_spec(module_name) is not None:
            print(f"Already available: {module_name}")
            continue

        try:
            subprocess.check_call([
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name,
            ])
            print(f"Installed: {package_name}")
        except subprocess.CalledProcessError as exc:
            print(f"Could not install optional package {package_name}: {exc}")
else:
    print("Skipping dependency installation.")

In [ ]:
import math
import random
import re
import tempfile
import pickle
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Some libraries still have non-deterministic behavior when using multiple workers,
# but fixing these seeds makes the scratch model and sampling steps more stable.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else None

print("Core libraries loaded.")

## 2. Dataset

For a fully standalone run, this notebook includes a small built-in book metadata dataset.

The dataset has the same kind of columns expected in this assignment:

- `title`
- `author`
- `description`
- `genres`
- `places`

A larger CSV can also be used by setting `USE_EXTERNAL_CSV = True`.

In [ ]:
CSV_PATH = Path("../Phase1/top_3000_rated_books.csv")


if CSV_PATH.exists():
    books_df = pd.read_csv(CSV_PATH)
    data_source = f"Loaded external dataset from {CSV_PATH}"

print(data_source)
print(f"Number of rows: {len(books_df)}")
books_df.head()

## 3. 🧹 Text Preprocessing

Before we can train word embedding models, we need to prepare the raw book metadata.

Book metadata usually contains a mixture of useful words, punctuation, repeated terms, numbers, and common words such as *the*, *and*, or *is*. These extra elements can make training less effective, so we first convert the raw text into a cleaner and more consistent format.
is Step

The goal of this section is to transform each book record into a clean list of meaningful tokens.

For example, a raw sentence like:

> A fantasy story about a young wizard, friendship, school, magic, and courage.

may become something like:

```text
["fantasy", "story", "young", "wizard", "friendship", "school", "magic", "courage"]

In [ ]:
# A compact English stopword list.
# These are common words that usually do not carry strong semantic meaning.
STOPWORDS = {
    "i", "me", "my", "myself", "we", "our", "ours", "ourselves",
    "you", "your", "yours", "yourself", "yourselves",
    "he", "him", "his", "himself", "she", "her", "hers", "herself",
    "it", "its", "itself", "they", "them", "their", "theirs", "themselves",
    "what", "which", "who", "whom", "this", "that", "these", "those",
    "am", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "having", "do", "does", "did", "doing",
    "a", "an", "the", "and", "but", "if", "or", "because", "as",
    "until", "while", "of", "at", "by", "for", "with", "about",
    "against", "between", "into", "through", "during", "before",
    "after", "above", "below", "to", "from", "up", "down", "in",
    "out", "on", "off", "over", "under", "again", "further",
    "then", "once", "here", "there", "when", "where", "why", "how",
    "all", "any", "both", "each", "few", "more", "most", "other",
    "some", "such", "no", "nor", "not", "only", "own", "same",
    "so", "than", "too", "very", "s", "t", "can", "will", "just",
    "don", "should", "now",
}

# Dataset-specific stopwords.
# These words appear because of the structure of the book metadata, not because
# they are semantically useful for learning word relationships.
DOMAIN_STOPWORDS = {
    "title", "titl",
    "author",
    "description", "descript",
    "genre", "genres", "genr",
    "place", "places",
    "goodreads", "goodread",
    "page", "pages",
    "book", "books",
    "edition", "editions",
    "isbn",
    "cover",
    "paperback",
    "hardcover",
    "kindle",
    "ebook",
}

# Final stopword set used during preprocessing.
ALL_STOPWORDS = STOPWORDS | DOMAIN_STOPWORDS


USE_STEMMING = False
stemmer = None

# Lemmatization is optional. The notebook runs without downloading external
# NLTK resources; WordNet is used only when it is already installed.
USE_LEMMATIZATION = True

lemmatizer = None
WORDNET_AVAILABLE = False

if USE_LEMMATIZATION:
    try:
        from nltk.stem import WordNetLemmatizer

        candidate_lemmatizer = WordNetLemmatizer()
        # This call verifies that the WordNet corpus itself is available.
        candidate_lemmatizer.lemmatize("books")
        lemmatizer = candidate_lemmatizer
        WORDNET_AVAILABLE = True
    except Exception:
        lemmatizer = None
        WORDNET_AVAILABLE = False
        print("WordNet is not available. Continuing without lemmatization.")


def normalize_token(token):
    """
    Normalize one token.

    Aggressive stemming is disabled by default so the learned vocabulary
    remains easy to inspect and interpret.
    """
    token = str(token).lower().strip()

    if USE_LEMMATIZATION and lemmatizer is not None:
        try:
            token = lemmatizer.lemmatize(token)
        except Exception:
            pass

    if USE_STEMMING and stemmer is not None:
        try:
            token = stemmer.stem(token)
        except Exception:
            pass

    return token


print(f"Number of general stopwords: {len(STOPWORDS)}")
print(f"Number of domain stopwords: {len(DOMAIN_STOPWORDS)}")
print(f"Total stopwords used: {len(ALL_STOPWORDS)}")
print(f"Stemming enabled: {USE_STEMMING}")
print(f"Lemmatization enabled: {USE_LEMMATIZATION and WORDNET_AVAILABLE}")

In [ ]:
class DataPreprocessor:
    """
    Converts book metadata into a clean tokenized corpus.

    Each row of the dataset becomes one cleaned sentence.
    The resulting token lists are used by all embedding models in this notebook.
    """

    def __init__(self, stopwords=None):
        self.stopwords = stopwords if stopwords is not None else set()

    def normalize_token(self, token):
        """
        Normalize one token using the notebook's selected normalization tools.
        """
        if lemmatizer is not None:
            try:
                token = lemmatizer.lemmatize(token)
            except Exception:
                pass

        if stemmer is not None:
            try:
                token = stemmer.stem(token)
            except Exception:
                pass

        return token

    def clean_text(self, text):
        """
        Clean one string and return one cleaned string.

        Important choices:
        - only English letters are kept,
        - words shorter than 3 characters are removed,
        - stopwords are removed,
        - remaining words are normalized.
        """
        if pd.isna(text):
            return ""

        text = str(text).lower()
        text = re.sub(r"[^a-z\s]", " ", text)

        words = text.split()
        cleaned_words = []

        for word in words:
            if word in self.stopwords or len(word) <= 2:
                continue

            normalized_word = self.normalize_token(word)

            if (
                normalized_word
                and len(normalized_word) > 2
                and normalized_word not in self.stopwords
            ):
                cleaned_words.append(normalized_word)

        return " ".join(cleaned_words)

    def build_corpus(self, df):
        """
        Combine metadata columns and return both:
        - cleaned text lines,
        - tokenized sentences.
        """
        required_columns = ["title", "author", "description", "genres", "places"]
        missing = [col for col in required_columns if col not in df.columns]

        if missing:
            raise ValueError(f"Missing required columns: {missing}")

        combined_text = (
            "title " + df["title"].fillna("").astype(str) + " "
            "author " + df["author"].fillna("").astype(str) + " "
            "description " + df["description"].fillna("").astype(str) + " "
            "genres " + df["genres"].fillna("").astype(str) + " "
            "places " + df["places"].fillna("").astype(str)
        )

        cleaned_lines = []
        tokenized_sentences = []

        for line in combined_text:
            cleaned = self.clean_text(line)

            if cleaned:
                cleaned_lines.append(cleaned)
                tokenized_sentences.append(cleaned.split())

        return cleaned_lines, tokenized_sentences

In [ ]:
preprocessor = DataPreprocessor(stopwords=ALL_STOPWORDS)

cleaned_lines, tokenized_sentences = preprocessor.build_corpus(books_df)

if not tokenized_sentences:
    raise ValueError("Preprocessing produced an empty corpus.")

print(f"Number of cleaned sentences: {len(cleaned_lines)}")
print("\nExample cleaned sentence:")
print(cleaned_lines[0])

The cleaned corpus is now ready for embedding training. Each sentence is represented as a list of tokens.

## 4. Vocabulary and Corpus Statistics

Before training, it is useful to inspect the vocabulary.

The vocabulary is the set of unique tokens in the cleaned corpus. Word embedding models learn one vector per vocabulary item.

In [ ]:
from collections import Counter

token_counts = Counter(
    token
    for sentence in tokenized_sentences
    for token in sentence
)

vocab = sorted(token_counts)
vocab_size = len(vocab)

print(f"Vocabulary size: {vocab_size}")
print("Most common tokens:")
token_counts.most_common(15)

In [ ]:
plt.figure(figsize=(10, 4))
top_words = token_counts.most_common(20)
plt.bar([word for word, count in top_words], [count for word, count in top_words])
plt.xticks(rotation=45, ha="right")
plt.title("Top 20 Tokens in the Cleaned Corpus")
plt.xlabel("Token")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

## 5. Embedding Visualization Helper

Word embeddings usually have many dimensions, such as 40 or 100. Since we cannot directly plot 40-dimensional or 100-dimensional vectors, we reduce them to 2 dimensions.

This notebook uses **PCA** when available. It is a simple SVD-based reduction.

In [ ]:
def reduce_to_2d(matrix):
    matrix = np.asarray(matrix, dtype=np.float64)

    if matrix.ndim != 2:
        raise ValueError("Input matrix must be a 2D array.")

    if matrix.shape[0] < 2:
        raise ValueError("At least two vectors are required for PCA.")

    mean_vector = np.mean(matrix, axis=0, keepdims=True)
    centered_matrix = matrix - mean_vector

    covariance_matrix = np.cov(centered_matrix, rowvar=False)

    covariance_matrix = np.atleast_2d(covariance_matrix)

    eigenvalues, eigenvectors = np.linalg.eigh(covariance_matrix)

    sorted_indices = np.argsort(eigenvalues)[::-1]
    top_component_indices = sorted_indices[:2]
    top_components = eigenvectors[:, top_component_indices]

    reduced_matrix = centered_matrix @ top_components

    if reduced_matrix.shape[1] == 1:
        reduced_matrix = np.column_stack(
            [reduced_matrix[:, 0], np.zeros(reduced_matrix.shape[0])]
        )

    return reduced_matrix


class VectorVisualizer:
    """
    Visualizes selected word vectors in a 2D plot.
    """

    def __init__(self, word_to_vec):
        self.word_to_vec = word_to_vec

    def visualize(self, target_words, title="2D Projection of Word Vectors"):
        vectors = []
        labels = []

        for word in target_words:
            if word in self.word_to_vec:
                vectors.append(
                    np.asarray(self.word_to_vec[word], dtype=np.float64)
                )
                labels.append(word)
            else:
                print(f"Warning: '{word}' was not found in the vocabulary.")

        if len(vectors) < 2:
            print("At least two known words are needed to create a plot.")
            return

        matrix = np.vstack(vectors)
        reduced = reduce_to_2d(matrix)

        plt.figure(figsize=(10, 7))
        plt.scatter(reduced[:, 0], reduced[:, 1])

        for i, label in enumerate(labels):
            plt.annotate(
                label,
                (reduced[i, 0], reduced[i, 1]),
                xytext=(5, 5),
                textcoords="offset points",
            )

        plt.title(title)
        plt.xlabel("Principal Component 1")
        plt.ylabel("Principal Component 2")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

## 6. Word2Vec from Scratch

This section implements a small skip-gram Word2Vec model from scratch.

### Skip-gram idea

For each center word, we try to predict nearby context words.

Example sentence:

```text
russian novel family history love
```

With a window size of 2, the center word `family` has context words:

```text
russian novel history love
```

### Negative sampling idea

Instead of comparing the center word against every word in the vocabulary, we train on:

- one real context word,
- several randomly sampled incorrect context words.

The model learns to assign a high score to real pairs and a low score to negative pairs.

In [ ]:
class SkipGramDataset:

    def __init__(self, sentences, window_size=2):
        self.sentences = [sentence for sentence in sentences if len(sentence) > 1]
        self.window_size = window_size

        if self.window_size < 1:
            raise ValueError("window_size must be at least 1.")

        self.build_vocab()
        self.generate_pairs()

    def build_vocab(self):
        vocab = sorted({
            word
            for sentence in self.sentences
            for word in sentence
        })

        self.word2idx = {
            word: idx
            for idx, word in enumerate(vocab)
        }
        self.idx2word = {
            idx: word
            for word, idx in self.word2idx.items()
        }
        self.vocab_size = len(vocab)

    def generate_pairs(self):
        """
        Generate center-context training pairs using the selected window size.
        """
        self.training_pairs = []

        for sentence in self.sentences:
            indexed_sentence = [self.word2idx[word] for word in sentence]

            for center_pos, center_word in enumerate(indexed_sentence):
                start = max(0, center_pos - self.window_size)
                end = min(
                    len(indexed_sentence),
                    center_pos + self.window_size + 1,
                )

                for context_pos in range(start, end):
                    if context_pos == center_pos:
                        continue

                    context_word = indexed_sentence[context_pos]
                    self.training_pairs.append((center_word, context_word))

    def get_negative_samples(self, positive_idx, k=5):
        if self.vocab_size < 2:
            raise ValueError("At least two vocabulary items are required.")

        negatives = []

        while len(negatives) < k:
            sample = int(np.random.randint(0, self.vocab_size))

            if sample != positive_idx:
                negatives.append(sample)

        return negatives

In [ ]:
class ScratchWord2Vec:

    def __init__(self, vocab_size, embedding_dim=100, learning_rate=0.025):
        if vocab_size < 1:
            raise ValueError("vocab_size must be positive.")
        if embedding_dim < 1:
            raise ValueError("embedding_dim must be positive.")

        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.learning_rate = learning_rate

        initialization_scale = 0.5 / embedding_dim
        self.W_in = np.random.uniform(
            -initialization_scale,
            initialization_scale,
            size=(vocab_size, embedding_dim),
        ).astype(np.float64)
        self.W_out = np.zeros(
            (vocab_size, embedding_dim),
            dtype=np.float64,
        )

    @staticmethod
    def sigmoid(x):
        x = np.clip(x, -15.0, 15.0)
        return 1.0 / (1.0 + np.exp(-x))

    def train_pair(self, center_idx, target_idx, negative_indices):
        """
        Train on one positive center-target pair and negative samples.
        """
        center_vec = self.W_in[center_idx].copy()
        target_vec = self.W_out[target_idx].copy()

        positive_score = np.dot(center_vec, target_vec)
        positive_pred = self.sigmoid(positive_score)
        positive_error = 1.0 - positive_pred

        center_gradient = positive_error * target_vec
        self.W_out[target_idx] += (
            self.learning_rate * positive_error * center_vec
        )

        for neg_idx in negative_indices:
            neg_vec = self.W_out[neg_idx].copy()

            negative_score = np.dot(center_vec, neg_vec)
            negative_pred = self.sigmoid(negative_score)
            negative_error = -negative_pred

            center_gradient += negative_error * neg_vec
            self.W_out[neg_idx] += (
                self.learning_rate * negative_error * center_vec
            )

        self.W_in[center_idx] += self.learning_rate * center_gradient

    def get_vector(self, idx):
        return self.W_in[idx]

    def save(self, path, word2idx, idx2word):
        data = {
            "vocab_size": self.vocab_size,
            "embedding_dim": self.embedding_dim,
            "learning_rate": self.learning_rate,
            "W_in": self.W_in,
            "W_out": self.W_out,
            "word2idx": word2idx,
            "idx2word": idx2word,
        }

        with open(path, "wb") as f:
            pickle.dump(data, f)

    @classmethod
    def load(cls, path):
        with open(path, "rb") as f:
            data = pickle.load(f)

        vocab_size = data["vocab_size"]
        embedding_dim = data["embedding_dim"]

        model = cls(
            vocab_size=vocab_size,
            embedding_dim=embedding_dim,
            learning_rate=data["learning_rate"],
        )
        model.W_in = np.asarray(data["W_in"], dtype=np.float64)
        model.W_out = np.asarray(data["W_out"], dtype=np.float64)
        model.word2idx = data["word2idx"]
        model.idx2word = data["idx2word"]

        return model

In [ ]:
class Trainer:


    def __init__(self, model, dataset, epochs=5, negative_samples=5, max_pairs=None):
        self.model = model
        self.dataset = dataset
        self.epochs = epochs
        self.negative_samples = negative_samples
        self.max_pairs = max_pairs

    def train(self):
        pairs = self.dataset.training_pairs

        if not pairs:
            raise ValueError("No skip-gram training pairs were generated.")

        if self.max_pairs is not None and self.max_pairs < len(pairs):
            pairs = random.sample(pairs, self.max_pairs)
            print(
                f"Training on a sampled subset of {len(pairs):,} pairs "
                f"out of {len(self.dataset.training_pairs):,}."
            )

        for epoch in range(self.epochs):
            epoch_pairs = list(pairs)
            random.shuffle(epoch_pairs)

            print(f"Epoch {epoch + 1}/{self.epochs}")

            for center_idx, target_idx in tqdm(epoch_pairs, leave=False):
                negative_indices = self.dataset.get_negative_samples(
                    positive_idx=target_idx,
                    k=self.negative_samples,
                )

                self.model.train_pair(
                    center_idx=center_idx,
                    target_idx=target_idx,
                    negative_indices=negative_indices,
                )

### Train the scratch Word2Vec model

The training settings below are intentionally small enough to run quickly in a notebook.

In [ ]:
scratch_dataset = SkipGramDataset(
    sentences=tokenized_sentences,
    window_size=3,
)

scratch_model = ScratchWord2Vec(
    vocab_size=scratch_dataset.vocab_size,
    embedding_dim=100,
    learning_rate=0.01,
)

scratch_trainer = Trainer(
    model=scratch_model,
    dataset=scratch_dataset,
    epochs=5,
    negative_samples=5,
    max_pairs=100_000,
)

print(f"Vocabulary size: {scratch_dataset.vocab_size}")
print(f"Number of skip-gram pairs: {len(scratch_dataset.training_pairs):,}")

scratch_start_time = perf_counter()
scratch_trainer.train()
scratch_elapsed_time = perf_counter() - scratch_start_time

print("Scratch Word2Vec training complete.")
print(f"Training time: {scratch_elapsed_time:.2f} seconds")

## 7. Testing Semantic Consistency with Odd-One-Out Questions

After training word embeddings, we need a simple way to inspect whether the model has learned meaningful relationships between words.


In each question, the model receives a small group of words. Most of the words should belong to the same semantic category, while one word should be less related to the others.

For example:

```text
fiction, novel, story, business

In [ ]:
class ScratchModelTester:

    def __init__(self, model, word2idx, idx2word):
        self.model = model
        self.word2idx = word2idx
        self.idx2word = idx2word

        self.word_to_vec = {
            word: self._to_numpy(self.model.get_vector(idx))
            for idx, word in self.idx2word.items()
        }

    def _to_numpy(self, vector):
        if hasattr(vector, "detach"):
            vector = vector.detach().cpu().numpy()

        return np.asarray(vector, dtype=np.float64).reshape(-1)

    def get_vector(self, word):
        if word not in self.word_to_vec:
            return None

        return self.word_to_vec[word]

    def cosine_similarity(self, vector_a, vector_b):
        vector_a = self._to_numpy(vector_a)
        vector_b = self._to_numpy(vector_b)

        norm_a = np.linalg.norm(vector_a)
        norm_b = np.linalg.norm(vector_b)

        if norm_a == 0 or norm_b == 0:
            return 0.0

        return float(np.dot(vector_a, vector_b) / (norm_a * norm_b))

    def find_similar_words(self, word, top_n=10):
        target_vector = self.get_vector(word)

        if target_vector is None:
            return f"'{word}' was not found in the vocabulary."

        similarities = []

        for other_word, other_vector in self.word_to_vec.items():
            if other_word == word:
                continue

            similarity = self.cosine_similarity(
                target_vector,
                other_vector,
            )
            similarities.append((other_word, similarity))

        similarities.sort(key=lambda item: item[1], reverse=True)

        return similarities[:top_n]

    def odd_one_out(self, words):
        if len(words) < 2:
            return {
                "prediction": None,
                "error": "At least two words are required.",
                "scores": None,
            }

        vectors = {}

        for word in words:
            vector = self.get_vector(word)

            if vector is None:
                return {
                    "prediction": None,
                    "error": f"'{word}' was not found in the vocabulary.",
                    "scores": None,
                }

            vectors[word] = vector

        scores = {}

        for word in words:
            similarities = []

            for other_word in words:
                if word == other_word:
                    continue

                similarity = self.cosine_similarity(
                    vectors[word],
                    vectors[other_word],
                )
                similarities.append(similarity)

            scores[word] = float(np.mean(similarities))

        prediction = min(scores, key=scores.get)

        return {
            "prediction": prediction,
            "error": None,
            "scores": scores,
        }

    def get_vector_dict(self):
        return self.word_to_vec

In [ ]:
odd_one_out_questions = [
    #Easy tests
    (["england", "london", "british", "dostoevsky"], "dostoevsky"),
    (["harry", "potter", "hogwarts", "tolstoy"], "tolstoy"),
    (["jane", "austen", "darcy", "wizard"], "wizard"),
    (["russia", "russian", "dostoevsky", "romance"], "romance"),
    (["america", "american", "united", "hogwarts"], "hogwarts"),
    (["france", "paris", "european", "vampire"], "vampire"),
    (["king", "queen", "lord", "british"], "british"),
    (["school", "student", "teacher", "tolstoy"], "tolstoy"),
    (["science", "fiction", "fantasy", "london"], "london"),
    (["mystery", "thriller", "crime", "austen"], "austen"),
    #Harder tests
    (["fiction", "novel", "story", "business"], "business"),
    (["fantasy", "magic", "fiction", "biography"], "biography"),
    (["mystery", "thriller", "crime", "poetry"], "poetry"),
    (["history", "historical", "century", "romance"], "romance"),
    (["biography", "memoir", "life", "horror"], "horror"),
    (["child", "young", "adult", "business"], "business"),
    (["austen", "fitzgerald", "miller", "travel"], "travel"),
    (["romance", "love", "family", "science"], "science"),
    (["music", "art", "poetry", "thriller"], "thriller"),
    (["business", "psychology", "self", "fantasy"], "fantasy"),
]

In [ ]:
scratch_tester = ScratchModelTester(
    model=scratch_model,
    word2idx=scratch_dataset.word2idx,
    idx2word=scratch_dataset.idx2word,
)

correct = 0
total = 0

for words, expected in odd_one_out_questions:
    result = scratch_tester.odd_one_out(words)

    print(f"Words: {words}")
    print(f"Expected odd word: {expected}")

    if result["error"] is not None:
        print(f"Model output: {result['error']}")
    else:
        prediction = result["prediction"]
        is_correct = prediction == expected

        correct += int(is_correct)
        total += 1

        print(f"Model output: {prediction}")
        print(f"Correct: {is_correct}")

        print("Scores, lower means more likely to be odd:")
        for word, score in sorted(result["scores"].items(), key=lambda item: item[1]):
            print(f"  {word:12s} {score:.4f}")

    print("-" * 60)

scratch_score = (correct, total)
print(f"Final score: {correct}/{total}")

## Model Performance Note

The scratch Word2Vec model has learned from the training corpus, but its results are still limited. This is expected because the dataset is relatively small and the model is trained from scratch without the optimizations used in larger embedding libraries.

Some predictions may be reasonable, while others may look noisy or unrelated. This does not mean the implementation is wrong; it shows that word embeddings need enough clean text and repeated context patterns to learn strong semantic relationships.

To improve the results, you can experiment with:

- increasing the number of training epochs,
- using a larger embedding dimension,
- cleaning the corpus more carefully,
- training on a larger dataset.

## 8. Visualizing Scratch Word2Vec Vectors

The plot below projects selected word vectors into 2D.

Words that appear near each other have vectors that are relatively similar under the learned embedding space.

In [ ]:
words_to_visualize = [
    "russian",
    "russia",
    "egypt",
    "africa",
    "america",
    "author",
    "friedrich",
    "albert",
    "rowl",
    "nietzsch",
    "camu",
    "jane",
    "austen",
]

In [ ]:
scratch_visualizer = VectorVisualizer(scratch_tester.get_vector_dict())
scratch_visualizer.visualize(
    target_words=words_to_visualize,
    title="Scratch Word2Vec: 2D Word Vector Projection",
)

## 9. FastText with Gensim

FastText improves on basic Word2Vec by using **subword information**.

Instead of learning only a vector for each whole word, FastText also learns from character n-grams. This helps the model handle:

- rare words,
- misspellings,
- related word forms,
- words that share roots.

This section trains a FastText model using Gensim.

In [ ]:
try:
    from gensim.models import FastText as GensimFastText
    from gensim.models.callbacks import CallbackAny2Vec

    GENSIM_AVAILABLE = True
except Exception as e:
    GensimFastText = None
    CallbackAny2Vec = None
    GENSIM_AVAILABLE = False
    print("Gensim is not available in this environment.")
    print("Error:", e)

print(f"Gensim available: {GENSIM_AVAILABLE}")

In [ ]:
gensim_fasttext_model = None
gensim_elapsed_time = None

if GENSIM_AVAILABLE:
    class TqdmEpochCallback(CallbackAny2Vec):
        """
        Show a tqdm progress bar during Gensim training.
        """

        def __init__(self, total_epochs):
            self.progress_bar = tqdm(
                total=total_epochs,
                desc="Training Gensim FastText",
            )

        def on_epoch_end(self, model):
            self.progress_bar.update(1)

        def on_train_end(self, model):
            self.progress_bar.close()


    GENSIM_VECTOR_SIZE = 40
    GENSIM_WINDOW_SIZE = 10
    GENSIM_MIN_COUNT = 1
    GENSIM_WORKERS = 1
    GENSIM_EPOCHS = 100

    gensim_fasttext_model = GensimFastText(
        vector_size=GENSIM_VECTOR_SIZE,
        window=GENSIM_WINDOW_SIZE,
        min_count=GENSIM_MIN_COUNT,
        workers=GENSIM_WORKERS,
        seed=RANDOM_SEED,
    )

    gensim_fasttext_model.build_vocab(
        corpus_iterable=tokenized_sentences
    )

    print("Starting Gensim FastText training...")
    print(f"Vocabulary size: {len(gensim_fasttext_model.wv.index_to_key)}")
    print(f"Epochs: {GENSIM_EPOCHS}")

    start_time = perf_counter()

    gensim_fasttext_model.train(
        corpus_iterable=tokenized_sentences,
        total_examples=gensim_fasttext_model.corpus_count,
        epochs=GENSIM_EPOCHS,
        callbacks=[TqdmEpochCallback(GENSIM_EPOCHS)],
    )

    gensim_elapsed_time = perf_counter() - start_time

    print("Gensim FastText training complete.")
    print(f"Training time: {gensim_elapsed_time:.2f} seconds")

else:
    print("Skipping Gensim FastText training because Gensim is unavailable.")

In [ ]:
class GensimFastTextTester:

    def __init__(self, model):
        self.model = model

        self.word_to_vec = {
            word: np.asarray(self.model.wv[word], dtype=np.float64)
            for word in self.model.wv.index_to_key
        }

    def get_vector(self, word):
        try:
            return np.asarray(self.model.wv[word], dtype=np.float64)
        except KeyError:
            return None

    def cosine_similarity(self, vector_a, vector_b):
        vector_a = np.asarray(vector_a, dtype=np.float64)
        vector_b = np.asarray(vector_b, dtype=np.float64)

        norm_a = np.linalg.norm(vector_a)
        norm_b = np.linalg.norm(vector_b)

        if norm_a == 0 or norm_b == 0:
            return 0.0

        return float(np.dot(vector_a, vector_b) / (norm_a * norm_b))

    def find_similar_words(self, word, top_n=10):
        try:
            return self.model.wv.most_similar(word, topn=top_n)
        except KeyError:
            return f"'{word}' was not found in the vocabulary."

    def odd_one_out(self, words):
        if len(words) < 2:
            return {
                "prediction": None,
                "error": "At least two words are required.",
                "scores": None,
            }

        vectors = {}

        for word in words:
            vector = self.get_vector(word)

            if vector is None:
                return {
                    "prediction": None,
                    "error": f"'{word}' was not found in the vocabulary.",
                    "scores": None,
                }

            vectors[word] = vector

        scores = {}

        for word in words:
            similarities = []

            for other_word in words:
                if word == other_word:
                    continue

                similarity = self.cosine_similarity(
                    vectors[word],
                    vectors[other_word],
                )
                similarities.append(similarity)

            scores[word] = float(np.mean(similarities))

        prediction = min(scores, key=scores.get)

        return {
            "prediction": prediction,
            "error": None,
            "scores": scores,
        }

    def get_vector_dict(self):
        return self.word_to_vec

In [ ]:
gensim_score = None

if gensim_fasttext_model is not None:
    gensim_tester = GensimFastTextTester(gensim_fasttext_model)

    correct = 0
    total = 0

    for words, expected in odd_one_out_questions:
        result = gensim_tester.odd_one_out(words)

        print(f"Words: {words}")
        print(f"Expected odd word: {expected}")

        if result["error"] is not None:
            print(f"Model output: {result['error']}")
        else:
            prediction = result["prediction"]
            is_correct = prediction == expected

            correct += int(is_correct)
            total += 1

            print(f"Model output: {prediction}")
            print(f"Correct: {is_correct}")

            print("Scores, lower means more likely to be odd:")
            for word, score in sorted(result["scores"].items(), key=lambda item: item[1]):
                print(f"  {word:12s} {score:.4f}")

        print("-" * 60)

    gensim_score = (correct, total)
    print(f"Final score: {correct}/{total}")

    gensim_visualizer = VectorVisualizer(gensim_tester.get_vector_dict())
    gensim_visualizer.visualize(
        target_words=words_to_visualize,
        title="Gensim FastText: 2D Word Vector Projection",
    )

else:
    print("No Gensim FastText model is available to test.")

## 10. FastText with the Native `fasttext` Package

The native FastText implementation expects text input as lines of tokens.

Since this notebook already has cleaned lines in memory, we create a temporary text corpus during this cell and use it for training.

This section is optional because the `fasttext` package can require compilation on some systems.

In [ ]:
try:
    import fasttext

    NATIVE_FASTTEXT_AVAILABLE = True
except Exception as e:
    fasttext = None
    NATIVE_FASTTEXT_AVAILABLE = False
    print("The native fasttext package is not available in this environment.")
    print("Error:", e)

print(f"Native fasttext available: {NATIVE_FASTTEXT_AVAILABLE}")

In [ ]:
from threading import Thread, Event
from time import sleep
import os

native_fasttext_model = None
native_elapsed_time = None

if NATIVE_FASTTEXT_AVAILABLE:
    NATIVE_VECTOR_SIZE = 40
    NATIVE_WINDOW_SIZE = 10
    NATIVE_EPOCHS = 25
    NATIVE_MIN_COUNT = 1

    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        suffix=".txt",
        delete=False,
    ) as tmp:
        tmp.write("\n".join(cleaned_lines))
        temp_corpus_path = tmp.name

    stop_progress = Event()

    def show_training_progress():
        with tqdm(desc="Training Native FastText", unit="sec") as progress_bar:
            while not stop_progress.wait(timeout=1):
                progress_bar.update(1)

    progress_thread = Thread(
        target=show_training_progress,
        daemon=True,
    )
    progress_thread.start()

    print("Starting native FastText training...")
    print(f"Vector size: {NATIVE_VECTOR_SIZE}")
    print(f"Window size: {NATIVE_WINDOW_SIZE}")
    print(f"Epochs: {NATIVE_EPOCHS}")
    print(f"Minimum word count: {NATIVE_MIN_COUNT}")

    start_time = perf_counter()

    try:
        native_fasttext_model = fasttext.train_unsupervised(
            input=temp_corpus_path,
            model="skipgram",
            dim=NATIVE_VECTOR_SIZE,
            ws=NATIVE_WINDOW_SIZE,
            epoch=NATIVE_EPOCHS,
            minCount=NATIVE_MIN_COUNT,
            thread=1,
            verbose=0,
        )
    finally:
        stop_progress.set()
        progress_thread.join()

        if os.path.exists(temp_corpus_path):
            os.remove(temp_corpus_path)

    native_elapsed_time = perf_counter() - start_time

    print("Native FastText training complete.")
    print(f"Training time: {native_elapsed_time:.2f} seconds")
    print(f"Vocabulary size: {len(native_fasttext_model.get_words())}")

else:
    print("Skipping native FastText training because the package is unavailable.")

In [ ]:
class NativeFastTextTester:

    def __init__(self, model):
        self.model = model
        self.words = self.model.get_words()

        self.word_to_vec = {
            word: np.asarray(
                self.model.get_word_vector(word),
                dtype=np.float64,
            )
            for word in self.words
        }

    def get_vector(self, word):
        if word not in self.word_to_vec:
            return None

        return self.word_to_vec[word]

    def cosine_similarity(self, vector_a, vector_b):
        vector_a = np.asarray(vector_a, dtype=np.float64)
        vector_b = np.asarray(vector_b, dtype=np.float64)

        norm_a = np.linalg.norm(vector_a)
        norm_b = np.linalg.norm(vector_b)

        if norm_a == 0 or norm_b == 0:
            return 0.0

        return float(np.dot(vector_a, vector_b) / (norm_a * norm_b))

    def find_similar_words(self, word, top_n=10):
        target_vector = self.get_vector(word)

        if target_vector is None:
            return f"'{word}' was not found in the vocabulary."

        similarities = []

        for other_word, other_vector in self.word_to_vec.items():
            if other_word == word:
                continue

            similarity = self.cosine_similarity(
                target_vector,
                other_vector,
            )
            similarities.append((other_word, similarity))

        similarities.sort(key=lambda item: item[1], reverse=True)

        return similarities[:top_n]

    def odd_one_out(self, words):
        if len(words) < 2:
            return {
                "prediction": None,
                "error": "At least two words are required.",
                "scores": None,
            }

        vectors = {}

        for word in words:
            vector = self.get_vector(word)

            if vector is None:
                return {
                    "prediction": None,
                    "error": f"'{word}' was not found in the vocabulary.",
                    "scores": None,
                }

            vectors[word] = vector

        scores = {}

        for word in words:
            similarities = []

            for other_word in words:
                if word == other_word:
                    continue

                similarity = self.cosine_similarity(
                    vectors[word],
                    vectors[other_word],
                )
                similarities.append(similarity)

            scores[word] = float(np.mean(similarities))

        prediction = min(scores, key=scores.get)

        return {
            "prediction": prediction,
            "error": None,
            "scores": scores,
        }

    def get_vector_dict(self):
        return self.word_to_vec

In [ ]:
native_score = None

if native_fasttext_model is not None:
    native_tester = NativeFastTextTester(native_fasttext_model)

    correct = 0
    total = 0

    for words, expected in odd_one_out_questions:
        result = native_tester.odd_one_out(words)

        print(f"Words: {words}")
        print(f"Expected odd word: {expected}")

        if result["error"] is not None:
            print(f"Model output: {result['error']}")
        else:
            prediction = result["prediction"]
            is_correct = prediction == expected

            correct += int(is_correct)
            total += 1

            print(f"Model output: {prediction}")
            print(f"Correct: {is_correct}")

            print("Scores, lower means more likely to be odd:")
            for word, score in sorted(result["scores"].items(), key=lambda item: item[1]):
                print(f"  {word:12s} {score:.4f}")

        print("-" * 60)

    native_score = (correct, total)
    print(f"Final score: {correct}/{total}")

    native_visualizer = VectorVisualizer(native_tester.get_vector_dict())
    native_visualizer.visualize(
        target_words=words_to_visualize,
        title="Native FastText: 2D Word Vector Projection",
    )

else:
    print("No native FastText model is available to test.")

## 11. Optional: Save and Reload the Scratch Model

Saving a trained model is useful when training takes a long time and we want to reuse the model later.

This section demonstrates saving and reloading the scratch Word2Vec model.

In [ ]:
scratch_model_path = Path("scratch_word2vec_answer.model")

scratch_model.save(
    path=scratch_model_path,
    word2idx=scratch_dataset.word2idx,
    idx2word=scratch_dataset.idx2word,
)

reloaded_scratch_model = ScratchWord2Vec.load(scratch_model_path)

print(f"Scratch model saved and reloaded from: {scratch_model_path}")
print(f"Reloaded embedding shape: {reloaded_scratch_model.W_in.shape}")

## 12. Model Comparison

The scratch Skip-Gram Word2Vec model is the easiest to understand because it clearly shows how embeddings are learned using center-context word pairs, negative samples, sigmoid scores, and gradient updates. However, it is slower, less efficient, and cannot create vectors for words that are not in its vocabulary. Gensim FastText is faster and more flexible because it learns from both full words and parts of words, which helps similar word forms share information and allows it to create vectors for many new words. Native fastText uses the same idea of learning from word parts and is built in compiled code, but differences in its settings and how it works can lead to different training speeds and results, even when similar settings are used.

In [ ]:
def format_score(score):
    if score is None:
        return "Not run"
    correct, total = score
    return f"{correct}/{total}"


comparison_rows = [
    {
        "Model": "Scratch Skip-Gram Word2Vec",
        "Status": "Completed",
        "Vocabulary size": scratch_dataset.vocab_size,
        "Vector size": scratch_model.embedding_dim,
        "Training seconds": round(scratch_elapsed_time, 3),
        "Odd-one-out score": format_score(scratch_score),
        "Uses subwords": False,
        "Supports OOV words": False,
    },
    {
        "Model": "Gensim FastText",
        "Status": "Completed" if gensim_fasttext_model is not None else "Skipped",
        "Vocabulary size": (
            len(gensim_fasttext_model.wv.index_to_key)
            if gensim_fasttext_model is not None
            else None
        ),
        "Vector size": (
            gensim_fasttext_model.vector_size
            if gensim_fasttext_model is not None
            else None
        ),
        "Training seconds": (
            round(gensim_elapsed_time, 3)
            if gensim_elapsed_time is not None
            else None
        ),
        "Odd-one-out score": format_score(gensim_score),
        "Uses subwords": True,
        "Supports OOV words": True,
    },
    {
        "Model": "Native fastText",
        "Status": "Completed" if native_fasttext_model is not None else "Skipped",
        "Vocabulary size": (
            len(native_fasttext_model.get_words())
            if native_fasttext_model is not None
            else None
        ),
        "Vector size": (
            native_fasttext_model.get_dimension()
            if native_fasttext_model is not None
            else None
        ),
        "Training seconds": (
            round(native_elapsed_time, 3)
            if native_elapsed_time is not None
            else None
        ),
        "Odd-one-out score": format_score(native_score),
        "Uses subwords": True,
        "Supports OOV words": True,
    },
]

comparison_df = pd.DataFrame(comparison_rows)
comparison_df